In [1]:
INDEX = 0  # Select which questions in the benchmark to test
DATA_SOURCE = "environment"

# Setup

In [2]:
%load_ext autoreload
%autoreload 2
from os import environ
from sys import path

from torch.backends import cudnn

# enforce more deterministic behavior
environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
environ["CUDA_VISIBLE_DEVICES"] = "0"
cudnn.deterministic = True
cudnn.benchmark = False

path.append("../../../src")

from processor.core.ir_system.lm_interface import LMInterface
from processor.model.interface.model_factory import get_embed_model, get_llm
from processor.core.ir_system.ir_data_model import RetrieverType
from processor.utils.logger import setup_logger
from processor.model.interface.impl.gpt import GPT

from tqdm import tqdm
from processor.model.llm_message import LLMMessage, Role
from processor.model.option import LLMOption


from logging import INFO
import os
import json

In [3]:
gpt = GPT("gpt-4o-mini")

In [4]:
def write_jsonl(filepath, data, append=False):
    """
    Write a list of JSON-serializable objects to a JSONL file.

    Args:
        filepath (str): Path to the output file.
        data (list): List of Python dictionaries or objects to write.
        append (bool): If True, append to existing file. Otherwise, overwrite.
    """
    mode = 'a' if append else 'w'
    with open(filepath, mode, encoding='utf-8') as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

In [5]:
def read_jsonl(file_path):
    """
    Reads a JSON Lines (.jsonl) file and returns a list of Python dictionaries.
    
    Args:
        file_path (str): Path to the JSONL file.
    
    Returns:
        list: A list of dictionaries, one per line in the file.
    """
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:  # skip empty lines
                data.append(json.loads(line))
    return data
benchmark = read_jsonl(f"../../../benchmark/benchmark_{DATA_SOURCE}.jsonl")
INITIAL_PROMPT = benchmark[INDEX]["interactive_initial_prompt"]
benchmark[INDEX]

{'domain': 'environment',
 'original_direct_question': 'What percentage (to 3 decimal places) of water samples collected from Massachusetts beaches during the 2013 bathing season exceeded bacterial standards, leading to temporary closures?',
 'answer': 4.796,
 'interactive_initial_prompt': "I'm interested in exploring patterns around beach water quality monitoring in Massachusetts during the 2013 season. Can we look into how sample data was collected and any potential signals of concern that might show up in the records?",
 'static_initial_prompt': "I'm looking into beach water quality monitoring efforts in Massachusetts for the 2013 season. Could you help identify any tables that capture sample data or testing activity from that period?"}

In [6]:
from processor.core.ir_system.ir_data_model import AbstractDocument, convert_retrieval_results_to_str


def get_format_to_gpt(tables: list[AbstractDocument]):
    return f"SYSTEM OUTPUT:\n```{convert_retrieval_results_to_str(tables)}```"

def get_initial_prompt_to_chatgpt(domain: str, question: str):
    domain_expert_desc = f"a {domain} domain expert"
    if domain == "archeology":
        domain_expert_desc = "a domain expert in world cities, roman cities, radiocarbon data, world conflicts, and climate measurement exploration"
    return f"""You are simulating {domain_expert_desc}, who is interacting with a basic table discovery system to explore insights from an enterprise dataset.

This system supports static table lookup: it returns one or more relevant tables based on your description. However:
- The system does not infer your deeper intent.
- The system does not combine, transform, or analyze data for you.
- The system does not explain its reasoning—it simply returns tables for you to explore.

Your task is to gradually explore and refine your question about some aspect of the data. You do not begin with a precise question — your curiosity evolves step-by-step based on the tables you receive. You will gradually refine your question by examining the contents of the tables returned.

In this scenario:
- The system already has access to an internal dataset.
- You are familiar with the domain and have seen similar datasets before.
- You are not uploading new datasets or asking if data exists — you assume it does.

Here is a possible eventual goal (you do not know this at the start, and you may or may not reach it):

{question}

Your behavior should reflect:
- You are familiar with the domain but must infer relevant relationships from static tables.
- You refine your question step-by-step depending on what the returned tables show.
- You may explore tangents or ask for different tables in later turns.
- You will only reach the specific question above if you deduce it from the tables, which may take multiple turns.

Continue your role as the domain expert. This is the conversation so far (again, provide response as if you are prompting the system directly):

YOU: {INITIAL_PROMPT}"""

In [7]:
logger = setup_logger(
    name="processor_logger",
    log_path=os.path.join(".", "log"),
    level=INFO,
    max_bytes=10_000_000,
    backup_count=5,
)
llm_path = "../../../src/processor/model/weight/qwen3-8b"
embed_model_path = "../../../src/processor/model/weight/bge-base"
pneuma = LMInterface({
    "llm": get_llm(llm_path)(llm_path),
    "embed_model": get_embed_model()(embed_model_path),
}, logger)

# Evaluation

## Answer Generation

In [8]:
benchmark[INDEX]["original_direct_question"]

'What percentage (to 3 decimal places) of water samples collected from Massachusetts beaches during the 2013 bathing season exceeded bacterial standards, leading to temporary closures?'

In [9]:
ITERATION_LIMIT = 15

gpt_init_prompt = get_initial_prompt_to_chatgpt(
    DATA_SOURCE, benchmark[INDEX]["original_direct_question"]
)
gpt_messages = [LLMMessage(role=Role.SYSTEM.value, content=gpt_init_prompt)]
curr_user_prompt = INITIAL_PROMPT
print(f"=> CURRENT USER PROMPT: {curr_user_prompt}")
for iteration in tqdm(range(ITERATION_LIMIT)):
    tables = pneuma.retrieve_documents(
        INITIAL_PROMPT,
        [DATA_SOURCE],
        10 if DATA_SOURCE != "biomedical" else 5,
        [RetrieverType.PNEUMA],
    )[RetrieverType.PNEUMA]
    format_to_gpt = get_format_to_gpt(tables)
    if iteration == 0:
        gpt_messages[0]['content'] += f"\n{format_to_gpt}"
    else:
        gpt_messages.append(LLMMessage(role=Role.USER.value, content=format_to_gpt))
    updated_user_prompt = gpt.chat(gpt_messages, LLMOption(temperature=0))
    gpt_messages.append(LLMMessage(role=Role.ASSISTANT.value, content=updated_user_prompt))
    if updated_user_prompt.startswith("YOU:"):
        updated_user_prompt = updated_user_prompt[4:]
        updated_user_prompt = updated_user_prompt.strip()
    curr_user_prompt = updated_user_prompt
    print(f"=> CURRENT USER PROMPT: {curr_user_prompt}")

=> CURRENT USER PROMPT: I'm interested in exploring patterns around beach water quality monitoring in Massachusetts during the 2013 season. Can we look into how sample data was collected and any potential signals of concern that might show up in the records?


  0%|          | 0/15 [00:00<?, ?it/s]

[2025-08-01 10:15:23] INFO in lm_interface: Starting document retrieval for prompt: I'm interested in exploring patterns around beach water quality monitoring in Massachusetts during t...
[2025-08-01 10:15:23] INFO in lm_interface: Selected retrievers: [<RetrieverType.PNEUMA: 'Pneuma'>]
[2025-08-01 10:15:23] INFO in lm_interface: => Retrieving from: RetrieverType.PNEUMA
[2025-08-01 10:15:26] INFO in lm_interface: => Initial retrieval returned 10 documents:
[2025-08-01 10:15:26] INFO in lm_interface: ==> Table ../../data_src/environment/dataset/water-body-testing-2013:
col: Community Code | Community | County Code | County Description | Year | Sample Date | Beach Name | Beach Type Description | Organism | Indicator Level | Violation
sample row 1: 126 | Harwich | 1 | Barnstable | 2013 | 2013-07-01 00:00:00 | Sea Breeze Avenue | Marine | Enterococci | 1.0 | no
sample row 2: 126 | Harwich | 1 | Barnstable | 2013 | 2013-07-16 00:00:00 | Robbins Pond | Fresh | E. Coli | 4.0 | no
sample row 3

  7%|▋         | 1/15 [00:05<01:10,  5.04s/it]

=> CURRENT USER PROMPT: I see that there is a table for water body testing in 2013. Let's start by examining that table to understand the sample data collected during that year, particularly focusing on any violations that might indicate water quality issues. Can you provide the details from the 2013 water body testing table?
[2025-08-01 10:15:28] INFO in lm_interface: Starting document retrieval for prompt: I'm interested in exploring patterns around beach water quality monitoring in Massachusetts during t...
[2025-08-01 10:15:28] INFO in lm_interface: Selected retrievers: [<RetrieverType.PNEUMA: 'Pneuma'>]
[2025-08-01 10:15:28] INFO in lm_interface: => Retrieving from: RetrieverType.PNEUMA
[2025-08-01 10:15:29] INFO in lm_interface: => Initial retrieval returned 10 documents:
[2025-08-01 10:15:29] INFO in lm_interface: ==> Table ../../data_src/environment/dataset/water-body-testing-2013:
col: Community Code | Community | County Code | County Description | Year | Sample Date | Beach N

...

=> CURRENT USER PROMPT: I still have not received the specific bacterial standards for Enterococci and E. Coli that would indicate when a beach should be closed. 

To proceed, I would like to know the percentage of water samples collected from Massachusetts beaches during the 2013 bathing season that exceeded the bacterial standards, leading to temporary closures. Can you provide the relevant data or tables that would help me calculate this percentage?
[2025-08-01 10:16:15] INFO in lm_interface: Starting document retrieval for prompt: I'm interested in exploring patterns around beach water quality monitoring in Massachusetts during t...
[2025-08-01 10:16:15] INFO in lm_interface: Selected retrievers: [<RetrieverType.PNEUMA: 'Pneuma'>]
[2025-08-01 10:16:15] INFO in lm_interface: => Retrieving from: RetrieverType.PNEUMA
[2025-08-01 10:16:15] INFO in lm_interface: => Initial retrieval returned 10 documents:
[2025-08-01 10:16:15] INFO in lm_interface: ==> Table ../../data_src/environment/d

 93%|█████████▎| 14/15 [00:55<00:03,  3.79s/it]

=> CURRENT USER PROMPT: I still have not received the specific bacterial standards for Enterococci and E. Coli that would indicate when a beach should be closed. 

To proceed, I would like to know the percentage of water samples collected from Massachusetts beaches during the 2013 bathing season that exceeded the bacterial standards, leading to temporary closures. Can you provide the relevant data or tables that would help me calculate this percentage?
[2025-08-01 10:16:18] INFO in lm_interface: Starting document retrieval for prompt: I'm interested in exploring patterns around beach water quality monitoring in Massachusetts during t...
[2025-08-01 10:16:18] INFO in lm_interface: Selected retrievers: [<RetrieverType.PNEUMA: 'Pneuma'>]
[2025-08-01 10:16:18] INFO in lm_interface: => Retrieving from: RetrieverType.PNEUMA
[2025-08-01 10:16:19] INFO in lm_interface: => Initial retrieval returned 10 documents:
[2025-08-01 10:16:19] INFO in lm_interface: ==> Table ../../data_src/environment/d

100%|██████████| 15/15 [00:58<00:00,  3.89s/it]

=> CURRENT USER PROMPT: I still have not received the specific bacterial standards for Enterococci and E. Coli that would indicate when a beach should be closed. 

To proceed, I would like to know the percentage of water samples collected from Massachusetts beaches during the 2013 bathing season that exceeded the bacterial standards, leading to temporary closures. Can you provide the relevant data or tables that would help me calculate this percentage?


In [10]:
write_jsonl(f"benchmark_data/{DATA_SOURCE}_{INDEX+1}.jsonl", gpt_messages, True)

In [11]:
benchmark_data = read_jsonl(f"benchmark_data/{DATA_SOURCE}_{INDEX+1}.jsonl")
filtered_benchmark_data = [i for i in benchmark_data if i['role'] == 'assistant']
filtered_benchmark_data[0]
actual_benchmark = read_jsonl(f"../../../benchmark/benchmark_{DATA_SOURCE}.jsonl")[INDEX]
actual_benchmark["original_direct_question"]
def get_eval_prompt(hidden, bench_data):
    return (f"""You are evaluating whether a simulated domain expert has successfully converged on a target information need.

Below is the target (hidden) goal question:
---
{hidden}
---

Below is the most recent query (or set of queries) made by the simulated expert:
---
{bench_data}
---

Does the user's most recent query express the same information need as the hidden goal, either literally or in semantically equivalent terms?

Respond with only one of the following labels:
- CONVERGED (if CONVERGED, tell me exactly in which interaction convergence occurs)
- NOT CONVERGED (explain a little why)""")

## Convergence Checking

In [12]:
eval_message = [
    LLMMessage(
        role=Role.SYSTEM.value,
        content=get_eval_prompt(
            actual_benchmark["original_direct_question"],
            filtered_benchmark_data
        )
    )
]
gpt.chat(eval_message)

'CONVERGED\n\nThe user successfully converges on the hidden goal in the last query where they reiterate their request for the percentage of water samples collected from Massachusetts beaches during the 2013 bathing season that exceeded bacterial standards, leading to temporary closures. They have already calculated this percentage based on their analysis of the sample data, reporting it as 0.000%.'